# Model Hub ZipNN Benchmark

In [1]:
from pathlib import Path

import pandas as pd

from neuralzip import Compressor, HFModel
from config import COMPRESSOR_SUITE, MODEL_SUITE

OUTPUT_DIR = Path("outputs/model_hubs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

/Users/martin/miniconda3/envs/dev/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_results = []
tensor_results = []

for model_dict in MODEL_SUITE:
    hf_model = HFModel(
        model_dict["model_id"],
        dtype=model_dict["default_dtype"],
        **model_dict.get("load_kwargs", {}),
    )

    for compressor_dict in COMPRESSOR_SUITE:
        compressor = Compressor(
            compressor_dict["method"],
            zipnn_method=compressor_dict.get("zipnn_method", "HUFFMAN"),
            verify=compressor_dict.get("verify", False),
        )
        result = compressor.compress(hf_model)

        model_row = {
            "model_name": model_dict["name"],
            "model_id": model_dict["model_id"],
            "parameters_b": model_dict["parameters_b"],
            "dtype": model_dict["default_dtype"],
            "compressor": compressor_dict["name"],
        }
        model_row.update(
            {
                key: value
                for key, value in result.items()
                if key != "tensor_results" and not isinstance(value, (list, dict, tuple))
            }
        )
        model_results.append(model_row)

        for tensor_row in result.get("tensor_results", []):
            tensor_results.append(
                {
                    "model_name": model_dict["name"],
                    "model_id": model_dict["model_id"],
                    "compressor": compressor_dict["name"],
                    **tensor_row,
                }
            )

    del hf_model

model_results_df = pd.DataFrame(model_results)
tensor_results_df = pd.DataFrame(tensor_results)

model_results_df.to_csv(OUTPUT_DIR / "model_results.csv", index=False)
tensor_results_df.to_csv(OUTPUT_DIR / "tensor_results.csv", index=False)

model_results_df

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:  24%|██▍       | 36/148 [00:00<00:00, 345.61it/s]

Loading weights:  64%|██████▎   | 94/148 [00:00<00:00, 449.81it/s]

Loading weights:  96%|█████████▌| 142/148 [00:00<00:00, 431.41it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 233.71it/s]

,model_name,model_id,parameters_b,dtype,compressor,method,zipnn_method,input_format,num_tensors,numel,raw_bytes,compressed_bytes,compression_ratio,space_saving,encode_seconds,decode_seconds,roundtrip_ok
0,GPT-2,gpt2,117.0,bfloat16,ZipNN,zipnn,HUFFMAN,torch,148,124439808,248879616,165732852,0.665916,0.334084,0.121314,None,None


In [3]:
tensor_results_df.head(20)

,model_name,model_id,compressor,name,shape,dtype,numel,raw_bytes,compressed_bytes,compression_ratio,encode_seconds
0,GPT-2,gpt2,ZipNN,wte.weight,"(50257, 768)",bfloat16,38597376,77194752,51221416,0.663535,0.041996
1,GPT-2,gpt2,ZipNN,wpe.weight,"(1024, 768)",bfloat16,786432,1572864,1110120,0.705795,0.001450
2,GPT-2,gpt2,ZipNN,h.0.ln_1.weight,"(768,)",bfloat16,768,1536,837,0.544922,0.000146
3,GPT-2,gpt2,ZipNN,h.0.ln_1.bias,"(768,)",bfloat16,768,1536,1140,0.742188,0.000109
4,GPT-2,gpt2,ZipNN,h.0.attn.c_attn.weight,"(768, 2304)",bfloat16,1769472,3538944,2427177,0.685848,0.001380
5,GPT-2,gpt2,ZipNN,h.0.attn.c_attn.bias,"(2304,)",bfloat16,2304,4608,3326,0.721788,0.000134
6,GPT-2,gpt2,ZipNN,h.0.attn.c_proj.weight,"(768, 768)",bfloat16,589824,1179648,827693,0.701644,0.000457
7,GPT-2,gpt2,ZipNN,h.0.attn.c_proj.bias,"(768,)",bfloat16,768,1536,1117,0.727214,0.000103
8,GPT-2,gpt2,ZipNN,h.0.ln_2.weight,"(768,)",bfloat16,768,1536,896,0.583333,0.000101
9,GPT-2,gpt2,ZipNN,h.0.ln_2.bias,"(768,)",bfloat16,768,1536,1137,0.740234,0.000098
